# NX-421 Mini-project — Part 1

Single-subject HCP data acquired during the language task (story vs math blocks).

- `data/` : raw dataset
- `derivatives/anat`, `derivatives/func` : every file produced
- `figures/` : PNGs used in the report and the video

Each processing step is skipped if its outputs already exist. If want to recompute everything : set `FORCE = True`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import os.path as op

sys.path.append(op.abspath(""))  

# Load FSL inside Neurodesk (as in the labs)
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
await lmod.list()

from utils import loadFSL          # utils.py copied from labs 
loadFSL()
os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"   # every FSL output as .nii.gz

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from nilearn import plotting, image

from helpers import (find_one, check_tools, run, needs, describe,
                     center_of_mass_world, tissue_volumes, spatial_corr, save_fig)

check_tools("fslreorient2std", "bet", "fast", "flirt", "fnirt", "applywarp")

In [ ]:
FORCE = False   # True = recompute every step even if its outputs already exist
ANAT_KW = dict(draw_cross=False, colorbar=False)   # common style for brain figures

PROJECT_ROOT = op.abspath("")      
DATA_DIR = op.join(PROJECT_ROOT, "data")
ANAT_DIR = op.join(PROJECT_ROOT, "derivatives", "anat")
FUNC_DIR = op.join(PROJECT_ROOT, "derivatives", "func")
FIG_DIR  = op.join(PROJECT_ROOT, "figures")
for d in (ANAT_DIR, FUNC_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

# Raw data (searched recursively)
T1_RAW   = find_one(DATA_DIR, "T1w.nii.gz")
FMRI_RAW = find_one(DATA_DIR, "tfMRI_LANGUAGE_LR.nii.gz")
EVENTS   = find_one(DATA_DIR, "tfMRI_LANGUAGE_LR_task_onsets.csv")

# MNI152 templates shipped with FSL
STD = op.join(os.environ["FSLDIR"], "data", "standard")
MNI_1MM            = op.join(STD, "MNI152_T1_1mm.nii.gz")
MNI_1MM_BRAIN      = op.join(STD, "MNI152_T1_1mm_brain.nii.gz")
MNI_1MM_BRAIN_MASK = op.join(STD, "MNI152_T1_1mm_brain_mask.nii.gz")
MNI_2MM            = op.join(STD, "MNI152_T1_2mm.nii.gz")
MNI_2MM_BRAIN      = op.join(STD, "MNI152_T1_2mm_brain.nii.gz")
MNI_2MM_MASK_DIL   = op.join(STD, "MNI152_T1_2mm_brain_mask_dil.nii.gz")

for p in (T1_RAW, FMRI_RAW, EVENTS):
    print(op.relpath(p, PROJECT_ROOT))

## 0. Look at the raw data 

Before the processing, we check the dimensions, voxel sizes and orientation of each image, and the task timing file.

Things to note for later:
- voxel size and number of volumes of the fMRI run (the 4th value is the TR);
- whether the fMRI grid already matches MNI 2 mm (91 × 109 × 91, 2 mm voxels), which matters for question 1.2c;
- whether the T1w orientation matches the MNI template.

In [ ]:
for p in (T1_RAW, FMRI_RAW, MNI_1MM):
    describe(p)
    print()

events = pd.read_csv(EVENTS)
events.head(10)

## 1.1 Structural preprocessing

### 1.1.0 Reorientation to the standard orientation

`fslreorient2std` only applies 90° rotations / flips so that the image axes follow the same convention as the MNI templates (no interpolation, no change of voxel values). This avoids registration starting from a wrongly oriented image, and every following step uses this single reoriented T1w, so the whole-head and skull-stripped images always share the same header (the issue lab 3 warns about with `fsl_anat` and `epi_reg`).

In [ ]:
T1 = op.join(ANAT_DIR, "T1w.nii.gz")   # reoriented whole-head T1w, used from now on

if needs(T1, force=FORCE):
    run(["fslreorient2std", T1_RAW, T1])
describe(T1)

### 1.1a Skull stripping (BET)

**Purpose.** Remove non-brain tissue (skull, scalp, eyes, neck) so that later steps only model brain tissue: FAST's intensity model would otherwise mix scalp and fat with brain classes, and linear registration to the MNI brain template is driven by the brain's shape rather than by the head.

**Method.** FSL BET. `-R` re-estimates the brain centre iteratively (useful when the field of view includes the neck), `-f` is the fractional intensity threshold (smaller → larger brain estimate), `-m` also saves the binary brain mask.

**QC.** Check that no brain is cut (frontal pole, superior parietal regions, cerebellum) and that no skull or dura remains. For a thorough check, open `T1w.nii.gz` and `T1w_brain_mask.nii.gz` in FSLeyes from the VM terminal and scroll through all slices.

In [ ]:
BET_F = 0.5     # if you change it, set FORCE = True (or delete the outputs) to re-run

T1_BRAIN      = op.join(ANAT_DIR, "T1w_brain.nii.gz")
T1_BRAIN_MASK = op.join(ANAT_DIR, "T1w_brain_mask.nii.gz")

if needs(T1_BRAIN, T1_BRAIN_MASK, force=FORCE):
    run(["bet", T1, T1_BRAIN.replace(".nii.gz", ""), "-R", "-f", BET_F, "-m"])

In [ ]:
cut = center_of_mass_world(T1_BRAIN_MASK)    # same slices in every panel

fig, axes = plt.subplots(3, 1, figsize=(11, 10))
plotting.plot_anat(T1, axes=axes[0], cut_coords=cut, **ANAT_KW,
                   title="Original T1w")
plotting.plot_anat(T1_BRAIN, axes=axes[1], cut_coords=cut, **ANAT_KW,
                   title=f"Skull-stripped T1w (BET, f = {BET_F})")
disp = plotting.plot_anat(T1, axes=axes[2], cut_coords=cut, **ANAT_KW,
                          title="Brain mask outline on the original T1w")
disp.add_contours(T1_BRAIN_MASK, levels=[0.5], colors="r", linewidths=1)
save_fig(fig, FIG_DIR, "1_1a_skull_stripping")
plt.show()

### 1.1b Tissue segmentation (FAST)

**Purpose.** Classify brain voxels into CSF, grey matter and white matter. The tissue maps are used later: GM to restrict or interpret activations, WM for the boundary-based registration of the fMRI to the T1w (`epi_reg`, lab 3), WM/CSF signals as possible nuisance regressors.

**Method.** FSL FAST on the skull-stripped image (as its documentation requires). It models each voxel's intensity as a mixture of tissue classes with a spatial prior (hidden Markov random field) while correcting the intensity bias field. `-t 1` = T1-weighted input, `-n 3` = three classes.

Outputs: `_pve_0/1/2` are partial-volume maps (fraction of each tissue per voxel), `_seg` is the hard segmentation. For a T1w, classes are ordered by intensity: `pve_0` / label 1 = CSF (darkest), `pve_1` / label 2 = GM, `pve_2` / label 3 = WM (brightest). Check it visually.

In [ ]:
FAST_BASE = op.join(ANAT_DIR, "T1w_fast")
SEG = FAST_BASE + "_seg.nii.gz"
PVE = {name: f"{FAST_BASE}_pve_{i}.nii.gz" for i, name in enumerate(["CSF", "GM", "WM"])}

if needs(SEG, *PVE.values(), force=FORCE):
    run(["fast", "-t", 1, "-n", 3, "-o", FAST_BASE, T1_BRAIN])   # a few minutes

In [ ]:
TISSUES = {1: ("CSF", "#3b8bd9"), 2: ("GM", "#e8743b"), 3: ("WM", "#f2d34f")}
seg_img = nib.load(SEG)

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
plotting.plot_anat(T1_BRAIN, axes=axes[0], cut_coords=cut, **ANAT_KW,
                   title="Skull-stripped T1w")
disp = plotting.plot_anat(T1_BRAIN, axes=axes[1], cut_coords=cut, **ANAT_KW,
                          title="FAST tissue segmentation")
for label, (name, color) in TISSUES.items():
    tissue = image.math_img(f"(img == {label}).astype('float32')", img=seg_img)
    disp.add_contours(tissue, levels=[0.5], filled=True, colors=[color])
fig.legend(handles=[Patch(color=c, label=n) for n, c in TISSUES.values()],
           loc="lower center", ncol=3, frameon=False)
save_fig(fig, FIG_DIR, "1_1b_segmentation")
plt.show()

# Sanity check: GM should be the largest class, then WM, then CSF.
# A very large CSF fraction often means BET left non-brain tissue.
vols = tissue_volumes(SEG, {lab: name for lab, (name, _) in TISSUES.items()})
pd.DataFrame(vols, index=["volume (mL)", "fraction"]).T.round(2)

### 1.1c Registration to MNI space (FLIRT + FNIRT)

**Purpose.** Bring the subject's anatomy into the standard MNI152 space so that results can be reported in standard coordinates, compared with atlases (e.g. the Yeo networks for 1.3e) and with the literature. The same transformation can later be reused to bring the functional data to MNI.

**Method** (FSL's recommended recipe, also what `fsl_anat` does):
1. **FLIRT**, affine (12 dof: translations, rotations, scalings, shears), skull-stripped T1w → MNI 2 mm brain. Captures global position, size and orientation.
2. **FNIRT**, non-linear, whole-head T1w → MNI 2 mm head (FNIRT expects images *with* skull, lab 2), initialised with the affine and using FSL's standard `T1_2_MNI152_2mm` configuration. Corrects local shape differences (ventricles, sulci).
3. **applywarp** resamples the skull-stripped T1w into MNI 1 mm with the combined transformation, for visualisation and as background for later overlays.

FNIRT can take 10–30 min on the VM.

In [ ]:
AFF        = op.join(ANAT_DIR, "T1w_to_MNI_affine.mat")
T1_MNI_LIN = op.join(ANAT_DIR, "T1w_brain_MNI_lin_2mm.nii.gz")
WARP       = op.join(ANAT_DIR, "T1w_to_MNI_warp.nii.gz")
T1_MNI     = op.join(ANAT_DIR, "T1w_brain_MNI.nii.gz")

# 1) Linear: affine, 12 dof
if needs(AFF, T1_MNI_LIN, force=FORCE):
    run(["flirt", "-in", T1_BRAIN, "-ref", MNI_2MM_BRAIN,
         "-omat", AFF, "-out", T1_MNI_LIN, "-dof", 12, "-cost", "corratio"])

# 2) Non-linear: FNIRT initialised with the affine
if needs(WARP, force=FORCE):
    run(["fnirt", f"--in={T1}", f"--aff={AFF}", "--config=T1_2_MNI152_2mm",
         f"--ref={MNI_2MM}", f"--refmask={MNI_2MM_MASK_DIL}", f"--cout={WARP}"])

# 3) Apply the warp to the skull-stripped T1w, on the 1 mm MNI grid
if needs(T1_MNI, force=FORCE):
    run(["applywarp", f"--ref={MNI_1MM_BRAIN}", f"--in={T1_BRAIN}",
         f"--warp={WARP}", f"--out={T1_MNI}", "--interp=spline"])

**QC.** Background: MNI152 template. Red: edges of the subject's brain. Before registration the subject image is only placed using its header (scanner coordinates); after registration, brain outline, ventricles and main sulci should follow the template. The correlation with the MNI brain inside the MNI brain mask gives a number to report alongside the figure.

In [ ]:
MNI_CUT = (0, -18, 12)   # MNI coordinates (mm), identical for all panels

stages = [
    (T1_BRAIN,   "Before registration (native T1w, header only)"),
    (T1_MNI_LIN, "After FLIRT (affine, 12 dof)"),
    (T1_MNI,     "After FLIRT + FNIRT (non-linear)"),
]

fig, axes = plt.subplots(len(stages), 1, figsize=(11, 10))
for ax, (img, title) in zip(axes, stages):
    disp = plotting.plot_anat(MNI_1MM, axes=ax, cut_coords=MNI_CUT,
                              **ANAT_KW, title=title)
    disp.add_edges(img, color="r")
save_fig(fig, FIG_DIR, "1_1c_registration_to_MNI")
plt.show()

for img, title in stages:
    r = spatial_corr(MNI_1MM_BRAIN, img, MNI_1MM_BRAIN_MASK)
    print(f"{title:<50s} r = {r:.3f}")

### Outputs of 1.1 reused later

| Variable | File | Used for |
|---|---|---|
| `T1`, `T1_BRAIN` | `T1w.nii.gz`, `T1w_brain.nii.gz` | fMRI → T1w coregistration (`epi_reg --t1 / --t1brain`) |
| `PVE["WM"]` | `T1w_fast_pve_2.nii.gz` | `epi_reg --wmseg` |
| `AFF`, `WARP` | `T1w_to_MNI_affine.mat`, `T1w_to_MNI_warp.nii.gz` | bringing fMRI to MNI (`applywarp --premat ... --warp ...`) if needed |
| `T1_MNI` | `T1w_brain_MNI.nii.gz` | background for the fMRI overlay in 1.2c |